# Isolamento florestal – MapBiomas Col. 11 (1986–2024), pipeline autogerenciável

Mesmo método do `isolation_forest_col11.js` (3 etapas por ano):
1. **100 m** – distâncias, fragmentos-alvo e distância mínima por fragmento → `isolation/forest_isolation_100m`
2. **30 m** – mapas para o paper/plataforma → `isolation/forest_isolation`
3. **Tabela** – CSV por ano no Drive (`PAPER-DEGRAD-COL11/isolation_forest_col11_<ano>.csv`)

O notebook lança a etapa 1 de todos os anos, e cada ano segue para as etapas 2 e 3 assim que a anterior termina. Ele monitora a cada 60 s, relança falhas (timeouts não) e pode ser re-executado: pula assets prontos, reanexa tasks rodando e não repete CSVs já concluídos.

**Se o Colab desconectar, as tasks continuam no GEE**: basta rodar as células de novo.

In [ ]:
!pip -q install -U earthengine-api
import ee
PROJECT = 'mapbiomas-brazil'   # Cloud project com quota de EE
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# módulo do repositório (branch col11)
!wget -q -O isolation3.py https://raw.githubusercontent.com/dh-conciani/forest-exposure-brazil/col11/gee/col11/isolation_3step/isolation3.py
import importlib, isolation3 as iso
importlib.reload(iso)

In [ ]:
cfg = iso.Config(
    distances=[500, 1000, 2000, 5000, 10000],   # m
    max_concurrent=1500,
    poll_seconds=60,
)
iso.check_inputs(cfg)

## 1) Teste com 1 ano (recomendado)
Veja quanto tempo a etapa 1 leva antes de soltar os 39 anos.

In [ ]:
jobs = iso.run(cfg, years=[2022])

## 2) Rodada completa (1986–2024)
Use `dry_run=True` para só ver o plano.

In [ ]:
iso.run(cfg, dry_run=True)

In [ ]:
jobs = iso.run(cfg)

## Log e checagem
`isolation3_log.jsonl` guarda início/fim/falha de cada task.

In [ ]:
import pandas as pd
log = pd.read_json(cfg.log_file, lines=True); log.tail(20)

In [ ]:
# quantos anos prontos em cada coleção
print(len(iso.list_ids(cfg.collection100)) // 2, 'anos em 100 m')
print(len(iso.list_ids(cfg.collection30)) // 2, 'anos em 30 m')